In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#Load dataset
df = pd.read_csv("netflix.csv")

df.head()
df.shape
df.info()

In [ ]:
#missing values
df.isnull().sum()

In [ ]:
#data cleaning
df["director"] = df["director"].fillna("Unknown")
df["cast"] = df["cast"].fillna("Unknown")
df["country"] = df["country"].fillna("Unknown")
df["rating"] = df["rating"].fillna(df["rating"].mode()[0])
df["duration"] = df["duration"].fillna("Unknown")

df["date_added"] = pd.to_datetime(df["date_added"].str.strip(), errors="coerce")
df["year_added"] = df["date_added"].dt.year
df["month_added"] = df["date_added"].dt.month_name()

In [ ]:
#remove duplicates
df.duplicated().sum()
df = df.drop_duplicates()

In [ ]:
#Statistics
df.describe()

In [ ]:
#Movies vs TV Shows
df["type"].value_counts()

sns.countplot(data=df, x="type")
plt.title("Movies vs TV Shows")
plt.show()

In [ ]:
#Content Rating Analysis
df["rating"].value_counts()

plt.figure(figsize=(10,5))
sns.countplot(data=df, x="rating", order=df["rating"].value_counts().index)
plt.xticks(rotation=45)
plt.title("Content Rating Distribution")
plt.show()

In [ ]:
#Genre Analysis
genre_df = df.assign(genre=df["listed_in"].str.split(", ")).explode("genre")
genre_df["genre"].value_counts().head(10)


plt.figure(figsize=(10,6))
genre_df["genre"].value_counts().head(10).plot(kind="barh")
plt.title("Top 10 Netflix Genres")
plt.xlabel("Count")
plt.show()


In [ ]:
#Release Year Analysis

df["release_year"].value_counts().head(10)

plt.figure(figsize=(12,5))
sns.histplot(df["release_year"], bins=30)
plt.title("Distribution of Content by Release Year")
plt.show()

In [ ]:
#Time Series Analysis
year_added = df["year_added"].value_counts().sort_index()

plt.figure(figsize=(12,5))
year_added.plot(kind="line", marker="o")
plt.title("Netflix Content Added Over Time")
plt.xlabel("Year Added")
plt.ylabel("Number of Titles")
plt.show()


In [ ]:
#Country Analysis
country_df = df.assign(country=df["country"].str.split(", ")).explode("country")
country_df["country"].value_counts().head(10)

plt.figure(figsize=(10,6))
country_df["country"].value_counts().head(10).plot(kind="barh")
plt.title("Top 10 Countries by Netflix Content")
plt.xlabel("Number of Titles")
plt.show()

In [ ]:
#Duration Analysis
df["duration_num"] = df["duration"].str.extract(r"(\d+)").astype(float)
df["duration_type"] = df["duration"].str.extract(r"(min|Season|Seasons)")

#Movie Duration
movie_df = df[df["type"] == "Movie"]
movie_df["duration_num"].mean()
movie_df["duration_num"].median()

sns.histplot(movie_df["duration_num"], bins=30, kde=True)
plt.title("Movie Duration Distribution")
plt.xlabel("Duration in Minutes")
plt.show()


In [ ]:
#TV Show Duration
tv_df = df[df["type"] == "TV Show"]
tv_df["duration_num"].mean()
tv_df["duration_num"].median()

In [ ]:
#Top Directors
df[df["director"] != "Unknown"]["director"].value_counts().head(10)

In [ ]:
#Top Cast
cast_df = df[df["cast"] != "Unknown"].assign(
    actor=df["cast"].str.split(", ")
).explode("actor")

cast_df["actor"].value_counts().head(10)

In [ ]:
#Genre Trends Over Time
genre_year = genre_df.groupby(["release_year", "genre"]).size().reset_index(name="count")
top_genres = genre_df["genre"].value_counts().head(5).index
genre_year_top = genre_year[genre_year["genre"].isin(top_genres)]

plt.figure(figsize=(12,6))
sns.lineplot(data=genre_year_top, x="release_year", y="count", hue="genre")
plt.title("Top Genre Trends Over Time")
plt.show()

In [ ]:
#Correlation Analysis
movie_df[["release_year", "duration_num"]].corr()

In [ ]:
#Content Variety
genre_df["genre"].nunique()

In [ ]:
#Word Cloud
from wordcloud import WordCloud
text = " ".join(df["description"].astype(str))
wordcloud = WordCloud(width=1000, height=500, background_color="black").generate(text)

plt.figure(figsize=(12,6))
plt.imshow(wordcloud)
plt.axis("off")
plt.title("Word Cloud of Netflix Descriptions")
plt.show()

###**Final Insights**
- Netflix has more Movies than TV Shows.
- Movies are about 69.62% of the dataset.
- TV Shows are about 30.38% of the dataset.
- Most content belongs to TV-MA and TV-14 ratings.
- The most common genre is International Movies.
- The United States has the highest Netflix content.
- India is the second-highest content-producing country.
- Most content was released around 2016–2020.
- Netflix added the highest number of titles in 2019.
- Average movie duration is around 100 minutes.
- Most TV shows have only 1 season.
- Netflix has 42 unique genres/categories.